<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/dnn-course-2026-1/blob/main/C1_M4_Lab_1_cnn_nature_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Construindo uma CNN para Classificação da Natureza

Suponha que o pessoal da área de biologia do campus nos procurou com uma ideia: um aplicativo que possa classificar não apenas flores, mas também insetos e pequenos animais. Sua tarefa é projetar e construir o modelo que tornará isso possível.

Este problema é mais complexo do que os que você enfrentou anteriormente. As camadas lineares que você utilizou antes não serão suficientes para capturar os ricos padrões visuais nestas diversas imagens. Para atender a este novo desafio, você construirá uma **Rede Neural Convolucional (CNN)**, um modelo projetado para reconhecer formas, texturas e características em dados visuais.

Neste laboratório, você passará pelo processo de ponta a ponta (*end-to-end*) de construção de uma CNN para esta tarefa de classificação. Você não apenas implementará a arquitetura, mas também seguirá um fluxo de trabalho iterativo — começando com um protótipo menor antes de aumentar a escala (*scaling up*) — e aprenderá a diagnosticar problemas comuns de treinamento.

Você irá:

* **Preparar um Conjunto de Dados Diversificado**: Carregar e transformar um subconjunto especializado de imagens para o seu classificador de natureza multiclasse.
* **Construir uma Arquitetura CNN**: Definir uma CNN completa do zero, combinando camadas convolucionais, de agrupamento (*pooling*) e totalmente conectadas (*fully connected*) para criar um poderoso extrator de características (*feature extractor*).
* **Treinar um Modelo Protótipo**: Seguir um fluxo de trabalho realista treinando primeiro o seu modelo em um subconjunto menor, de 9 classes, para construir um protótipo funcional e estabelecer uma linha de base (*baseline*) de desempenho.
* **Aumentar a Escala e Diagnosticar Desafios**: Treinar o modelo completo em todas as 15 classes e analisar os resultados para identificar desafios comuns de aprendizado de máquina, como o *overfitting* (sobreajuste).

## Importação de bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import helper_utils

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Preparando o Conjunto de Dados da Natureza

Para este laboratório, você trabalhará com uma coleção de imagens retiradas do conhecido [conjunto de dados CIFAR-100](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.CIFAR100.html). Este conjunto de dados é um recurso fantástico para tarefas de visão computacional, contendo milhares de pequenas **imagens coloridas de 32x32**, que são perfeitas para treinar uma CNN. É uma coleção diversificada, o que é exatamente o que você precisa para o aplicativo expandido de classificação da natureza.

Embora o CIFAR-100 tenha 100 classes diferentes, você não precisará de todas elas. Para atender aos novos requisitos do seu aplicativo, você se concentrará em uma seleção curada de **15 classes** que se encaixam no tema de um classificador de natureza. Esta seleção incluirá flores, insetos e mamíferos. Especificamente, você trabalhará com:

* **Flores** (*Flowers*): 'orchid', 'poppy', 'rose', 'sunflower', 'tulip'
* **Mamíferos** (*Mammals*): 'fox', 'porcupine', 'possum', 'raccoon', 'skunk'
* **Insetos** (*Insects*): 'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'

### Transformações de Imagem

Antes de carregar o conjunto de dados, primeiro você precisa definir os pipelines de transformação para ele. Como todas as imagens no conjunto de dados já possuem um tamanho padrão de **32x32**, você não precisa adicionar uma etapa de redimensionamento. Seu pipeline de treinamento incluirá aumento de dados (*data augmentation*), enquanto ambos os pipelines converterão as imagens em **tensores** e as **normalizarão** usando os valores padrão de média e desvio padrão para o conjunto de dados CIFAR-100.

* Defina a média e o desvio padrão específicos para o conjunto de dados CIFAR-100.

In [ ]:
cifar100_mean = (0.5071, 0.4867, 0.4408)
cifar100_std = (0.2675, 0.2565, 0.2761)

* Defina dois pipelines separados utilizando `transforms.Compose`.
    * Um para o conjunto de treinamento, que inclui aumento de dados (*data augmentation*), e outro para o conjunto de validação.

In [ ]:
# Training set transformation pipeline
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

# Validation set transformation pipeline
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar100_mean, cifar100_std)
])

### Preparando o Pipeline de Dados

Com suas transformações prontas, é hora de carregar os dados. Para mostrar rapidamente um protótipo funcional ao borboletário vizinho, uma estratégia inteligente é começar com um conjunto de dados menor e mais gerenciável. Isso permite que você teste todo o seu pipeline e construa um modelo de linha de base (*baseline*) sem os longos tempos de espera exigidos para o conjunto de dados completo.

Portanto, em vez de usar todas as 15 classes de uma vez, você começará com um subconjunto balanceado de **9 classes** (**3 de cada categoria**). Essa abordagem iterativa é uma prática comum e eficiente no aprendizado de máquina do mundo real.

Para este protótipo inicial, você usará as seguintes classes:

* **Flores** (*Flowers*): 'orchid', 'poppy', 'sunflower'
* **Mamíferos** (*Mammals*): 'fox', 'raccoon', 'skunk'
* **Insetos** (*Insects*): 'butterfly', 'caterpillar', 'cockroach'

* Crie uma lista em Python contendo os nomes das 9 classes que você usará para o protótipo inicial.

In [ ]:
subset_target_classes = [
    # Flowers
    'orchid', 'poppy', 'sunflower',
    # Mammals
    'fox', 'raccoon', 'skunk',
    # Insects
    'butterfly', 'caterpillar', 'cockroach'
]

* Use a função auxiliar `load_cifar100_subset`, passando a sua lista `subset_target_classes` e ambos os pipelines de transformação.
* Esta função lida com todo o processo de carregamento: ela faz o download do conjunto de dados CIFAR-100 completo, aplica as transformações especificadas e, em seguida, filtra o resultado para incluir apenas as **9 classes** que você selecionou.
* Ela retorna os objetos finais dos conjuntos de dados de treinamento e validação, prontos para a próxima etapa.

In [ ]:
# Call the helper function to prepare the datasets
train_dataset_proto, val_dataset_proto = helper_utils.load_cifar100_subset(subset_target_classes, train_transform, val_transform)

* Com seus objetos `Dataset` prontos, a etapa final no pipeline de dados é criar os `DataLoaders`.

In [ ]:
# Set the number of samples to be processed in each batch
batch_size = 64

# Create a data loader for the training set, with shuffling enabled
train_loader_proto = DataLoader(train_dataset_proto, batch_size=batch_size, shuffle=True)

# Create a data loader for the validation set, without shuffling
val_loader_proto = DataLoader(val_dataset_proto, batch_size=batch_size, shuffle=False)

### Visualizando as Imagens de Treinamento

Com o seu pipeline de dados concluído, é sempre uma boa ideia analisar alguns exemplos do seu conjunto de treinamento. Isso ajuda a confirmar se os seus dados foram carregados e processados corretamente. A função auxiliar a seguir exibirá uma amostra aleatória das suas imagens de treinamento.

In [ ]:
# Visualize a 3x3 grid of random training images
helper_utils.visualise_images(train_dataset_proto, grid=(3, 3))

## Construindo a Arquitetura da CNN

Com seus dados prontos, é hora de construir o núcleo do seu classificador de natureza. Para uma tarefa complexa como a identificação de diferentes espécies em imagens, as camadas lineares que você usou antes não são suficientes, pois elas analisam os pixels individualmente sem entender suas relações espaciais.

Agora você construirá uma **Rede Neural Convolucional (CNN)**, uma arquitetura projetada especificamente para "ver" e reconhecer padrões, bordas e texturas em imagens através de uma série de filtros aprendíveis (*learnable filters*). Você definirá a estrutura do seu modelo usando o `nn.Module` do PyTorch, combinando vários tipos de camadas para criar um poderoso classificador de imagens.

Aqui está um detalhamento das principais camadas que você usará:

**Camada Convolucional ([nn.Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html))**

> Este é o bloco de construção central de uma CNN, que usa filtros aprendíveis para varrer a imagem em busca de características visuais. A saída é um conjunto de "mapas de características" (*feature maps*) que destacam onde esses padrões aparecem na imagem.
> * `in_channels`: O número de canais da camada anterior; para a primeira camada, este valor é 3, correspondente aos canais de cores RGB.
> * `out_channels`: O número de filtros que a camada irá aprender, determinando o número de mapas de características de saída.
> * `kernel_size`: As dimensões do filtro, como uma grade 3x3 que examina um pixel e seus vizinhos imediatos.
> * `padding`: Adiciona uma borda ao redor da imagem, permitindo que o *kernel* processe os pixels da extremidade enquanto preserva as dimensões originais da imagem.
> 
> 

**Função de Ativação ReLU ([nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html))**

> Uma função de ativação que introduz não-linearidade mudando todos os valores negativos nos mapas de características para zero. Isso ajuda o modelo a aprender padrões mais complexos.

**Camada de Agrupamento Máximo (*Max Pooling*) ([nn.MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html))**

> Esta camada reduz a resolução (*downsamples*) dos mapas de características diminuindo sua altura e largura, o que torna a rede mais eficiente. Ela desliza uma janela sobre o mapa de características e mantém apenas o maior valor único daquela janela, descartando o resto.
> * `kernel_size`: O tamanho da janela na qual o agrupamento (*pooling*) será realizado, como uma área 2x2.
> * `stride`: O tamanho do passo que a janela se move através da imagem. Um *stride* de 2 com um *kernel* 2x2 reduzirá pela metade as dimensões do mapa de características.
> 
> 

**Camada de Achatamento (*Flatten*) ([nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.Flatten.html))**

> Uma camada utilitária que desenrola (*unrolls*) os mapas de características 2D em um único vetor 1D. Esta é uma etapa necessária para preparar os dados para as camadas lineares totalmente conectadas.

**Camada Linear ([nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html))**

> Também conhecida como camada totalmente conectada (*fully connected layer*), ela realiza a classificação final. Ela combina as características aprendidas pelas camadas convolucionais em uma previsão final.

**Camada de Abandono (*Dropout*) ([nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html))**

> Uma técnica de regularização que ajuda a evitar o *overfitting* (sobreajuste), definindo aleatoriamente uma fração das ativações dos neurônios como zero durante o treinamento. Isso força a rede a aprender características mais robustas em vez de depender muito de qualquer padrão único.

In [ ]:
class SimpleCNN(nn.Module):
    """
    A simple Convolutional Neural Network model.

    The architecture consists of three convolutional blocks followed by two
    fully connected layers for classification.
    """
    def __init__(self, num_classes):
        """
        Initializes the layers of the neural network.

        Args:
            num_classes: The number of output classes for the final layer.
        """
        # Call the constructor of the parent class (nn.Module)
        super(SimpleCNN, self).__init__()
        
        # Define the first convolutional block
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the second convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the third convolutional block
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Define the layer to flatten the feature maps
        self.flatten = nn.Flatten()

        # Define the fully connected (dense) layers
        # Input image is 32x32, after 3 pooling layers: 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)


    def forward(self, x):
        """
        Defines the forward pass of the model.

        Args:
            x: The input tensor of shape (batch_size, channels, height, width).

        Returns:
            The output tensor containing the logits for each class.
        """
        # Pass input through the first convolutional block
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Pass feature maps through the second convolutional block
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Pass feature maps through the third convolutional block
        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)

        # Flatten the output for the fully connected layers
        x = self.flatten(x)

        # Pass the flattened features through the fully connected layers
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout(x)
        x = self.fc2(x)

        # Return the final output logits
        return x

With the `SimpleCNN` architecture defined, the next step is to create an instance of the model for your prototype.

* First, dynamically determine the number of output classes by checking the length of the class list in your `train_dataset_proto`.
* Create an instance of your `SimpleCNN`, passing `num_classes` to its constructor.

In [ ]:
# Get the number of classes
num_classes = len(train_dataset_proto.classes)

# Instantiate the model
prototype_model = SimpleCNN(num_classes)

Before you start training, it's very helpful to visualize how the shape of your data changes as it flows through the CNN. This will confirm that your architecture is set up correctly and show you how the spatial dimensions shrink while the number of channels grows with each convolutional block.

* Define the `print_data_flow` helper function. 
    * This function will pass a sample 32x32 color image through your model, layer by layer, printing the tensor's shape at each key step to trace its journey from input to final prediction.

In [ ]:
def print_data_flow(model):
    """
    Prints the shape of a tensor as it flows through each layer of the model.

    Args:
        model: An instance of the PyTorch model to inspect.
    """
    # Create a sample input tensor (batch_size, channels, height, width)
    x = torch.randn(1, 3, 32, 32)

    # Track the tensor shape at each stage
    print(f"Input shape: \t\t{x.shape}")

    # First conv block
    x = model.conv1(x)
    print(f"After conv1: \t\t{x.shape}")
    x = model.relu1(x)
    x = model.pool1(x)
    print(f"After pool1: \t\t{x.shape}")

    # Second conv block
    x = model.conv2(x)
    print(f"After conv2: \t\t{x.shape}")
    x = model.relu2(x)
    x = model.pool2(x)
    print(f"After pool2: \t\t{x.shape}")

    # Third conv block
    x = model.conv3(x)
    print(f"After conv3: \t\t{x.shape}")
    x = model.relu3(x)
    x = model.pool3(x)
    print(f"After pool3: \t\t{x.shape}")

    # Flatten using the model's flatten layer
    x = model.flatten(x)
    print(f"After flatten: \t\t{x.shape}")

    # Fully connected layers
    x = model.fc1(x)
    print(f"After fc1: \t\t{x.shape}")
    x = model.relu4(x)
    x = model.dropout(x)
    x = model.fc2(x)
    print(f"Output shape (fc2): \t{x.shape}")

You can now print a summary of your model and trace the data flow to see it in action.

* Call your helper function to print the tensor's shape at each step.

    * The tensor starts as a `(1, 3, 32, 32)` image. As it passes through the `conv` and `pool` blocks, the number of **channels increases** while the **spatial size is halved** at each step.

    * The final `(1, 128, 4, 4)` feature map is **flattened** into a 1D vector to be processed by the linear layers. The model's final **output shape is** `(1, 9)`, providing one score for each of the 9 classes.

In [ ]:
# Print the model's architecture
print(prototype_model)

# Call the helper function to visualize the data flow
print("\n--- Tracing Data Flow ---")
print_data_flow(prototype_model)

## Training the Model

With your model defined and the data pipeline prepared, you're ready to set up the training process. This involves initializing a loss function to measure your model's error and an optimizer to update its weights based on that error.

### Initialize Loss Function and Optimizer

Before starting the training loop, you'll define two key components:

* You'll use <code>[nn.CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)</code>. This is the standard loss function for multi-class classification tasks as it's designed to measure the error when a model has to choose one class from several possibilities.
* You'll use the <code>[Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)</code> optimizer. This is a popular and efficient algorithm that updates the model's weights to minimize the loss.

In [ ]:
# Loss function
loss_function = nn.CrossEntropyLoss()

# Optimizer for the prototype model
optimizer_prototype = optim.Adam(prototype_model.parameters(), lr=0.001)

### The Training Loop

* Next, you'll define the `training_loop` function. This function encapsulates the entire process of training and validating your model over multiple epochs.

In [ ]:
def training_loop(model, train_loader, val_loader, loss_function, optimizer, num_epochs, device):
    """
    Trains and validates a PyTorch neural network model.

    Args:
        model: The neural network model to be trained.
        train_loader: DataLoader for the training dataset.
        val_loader: DataLoader for the validation dataset.
        loss_function: The loss function to use for training.
        optimizer: The optimization algorithm.
        num_epochs: The total number of epochs to train for.
        device: The device (e.g., 'cpu' or 'cuda') to run the training on.

    Returns:
        A tuple containing:
        - The trained model.
        - A list of metrics [train_losses, val_losses, val_accuracies].
    """
    # Move the model to the specified device (CPU or GPU)
    model.to(device)
    
    # Initialize lists to store training and validation metrics
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    # Print a message indicating the start of the training process
    print("--- Training Started ---")
    
    # Loop over the specified number of epochs
    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()
        # Initialize running loss for the current epoch
        running_loss = 0.0
        # Iterate over batches of data in the training loader
        for images, labels in train_loader:
            # Move images and labels to the specified device
            images, labels = images.to(device), labels.to(device)
            
            # Clear the gradients of all optimized variables
            optimizer.zero_grad()
            # Perform a forward pass to get model outputs
            outputs = model(images)
            # Calculate the loss
            loss = loss_function(outputs, labels)
            # Perform a backward pass to compute gradients
            loss.backward()
            # Update the model parameters
            optimizer.step()
            
            # Accumulate the training loss for the batch
            running_loss += loss.item() * images.size(0)
            
        # Calculate the average training loss for the epoch
        epoch_loss = running_loss / len(train_loader.dataset)
        # Append the epoch loss to the list of training losses
        train_losses.append(epoch_loss)
        
        # Set the model to evaluation mode
        model.eval()
        # Initialize running validation loss and correct predictions count
        running_val_loss = 0.0
        correct = 0
        total = 0
        # Disable gradient calculations for validation
        with torch.no_grad():
            # Iterate over batches of data in the validation loader
            for images, labels in val_loader:
                # Move images and labels to the specified device
                images, labels = images.to(device), labels.to(device)
                
                # Perform a forward pass to get model outputs
                outputs = model(images)
                
                # Calculate the validation loss for the batch
                val_loss = loss_function(outputs, labels)
                # Accumulate the validation loss
                running_val_loss += val_loss.item() * images.size(0)
                
                # Get the predicted class labels
                _, predicted = torch.max(outputs, 1)
                # Update the total number of samples
                total += labels.size(0)
                # Update the number of correct predictions
                correct += (predicted == labels).sum().item()
                
        # Calculate the average validation loss for the epoch
        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        # Append the epoch validation loss to the list
        val_losses.append(epoch_val_loss)
        
        # Calculate the validation accuracy for the epoch
        epoch_accuracy = 100.0 * correct / total
        # Append the epoch accuracy to the list
        val_accuracies.append(epoch_accuracy)
        
        # Print the metrics for the current epoch
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Accuracy: {epoch_accuracy:.2f}%")
        
    # Print a message indicating the end of the training process
    print("--- Finished Training ---")
    
    # Consolidate all metrics into a single list
    metrics = [train_losses, val_losses, val_accuracies]
    
    # Return the trained model and the collected metrics
    return model, metrics

With all the components in place, you're ready to start training.

* Run the `training_loop` function with your prototype model (for 9 classes), its corresponding data loaders, the loss function, and the optimizer. 
* You'll train for `15 epochs`, and the function will return the trained model along with the collected performance metrics.
* After training is complete, you'll use the `plot_training_metrics` helper function to visualize the training and validation loss, along with the validation accuracy. 

In [ ]:
# Start the training process by calling the training loop function
trained_proto_model, training_metrics_proto = training_loop(
    model=prototype_model, 
    train_loader=train_loader_proto, 
    val_loader=val_loader_proto, 
    loss_function=loss_function, 
    optimizer=optimizer_prototype, 
    num_epochs=15, 
    device=device
)

# Visualize the training metrics (loss and accuracy)
print("\n--- Training Plots ---\n")
helper_utils.plot_training_metrics(training_metrics_proto)

<br>

Excellent work! The prototype model is trained, and the results look very promising. Achieving a validation accuracy of over **75%** on the 9-class subset is a great result and confirms that your CNN architecture is well-suited for this task.

This successful prototype gives you the green light to move forward with the next phase: training a full-scale model on all 15 classes for the butterfly house. But before you do, it's helpful to perform one last qualitative check to see how your model "thinks."

### Visualizing Predictions

While the plots show your model's overall performance, looking at individual predictions provides a more intuitive feel for its strengths and weaknesses. You can now use a helper function to see your model in action, visualizing its predictions on random images from the validation set. This will show you concrete examples of where it succeeds and where it might be making mistakes.

In [ ]:
# Visualize model predictions on a sample of validation images
helper_utils.visualise_predictions(
    model=trained_proto_model, 
    data_loader=val_loader_proto, 
    device=device, 
    grid=(3, 3)
)

## Scaling Up: Training the Full Model 

The prototype was a success! Now it's time to train the final model for the butterfly house app. You'll repeat the same steps as before, but this time using the full, more challenging dataset of **15 classes**.

* First, create a new list containing all 15 target classes.
* Use the `load_cifar100_subset` helper function again to create the new training and validation datasets based on this full list.

In [ ]:
# Define the full class list.
all_target_classes = [
    # Flowers
    'orchid', 'poppy', 'rose', 'sunflower', 'tulip',
    # Mammals
    'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
    # Insects
    'bee', 'beetle', 'butterfly', 'caterpillar', 'cockroach'
]

# Load the full datasets.
train_dataset, val_dataset = helper_utils.load_cifar100_subset(all_target_classes, train_transform, val_transform)

<br>

* Wrap your new 15-class datasets in `DataLoader` instances, using the same `batch_size=64`.

In [ ]:
# Create a data loader for the training set, with shuffling enabled
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create a data loader for the validation set, without shuffling
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

* Display a sample of images from your new 15-class training set to confirm it has been loaded correctly.

In [ ]:
# Visualize a 3x5 grid of random training images
helper_utils.visualise_images(train_dataset, grid=(3, 5))

<br>

* Create a new instance of your `SimpleCNN` model, this time configured for all **15 classes**.

In [ ]:
# Get the number of classes
num_classes = len(train_dataset.classes)

# Instantiate the full model
model = SimpleCNN(num_classes)

# Print the model's architecture (notice, it now has 15 output classes)
print(model)

<br>

* Create a new `Adam` optimizer for your full 15-class model.

In [ ]:
# Optimizer for the full model
optimizer = optim.Adam(model.parameters(), lr=0.001)

* Call the `training_loop` to train your 15-class model for `25 epochs`. The `plot_training_metrics` function will then immediately visualize the loss and accuracy curves from this final training run.

In [ ]:
# Start the training process for the full model on all 15 classes
trained_model, training_metrics = training_loop(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    loss_function=loss_function, 
    optimizer=optimizer, 
    num_epochs=25, 
    device=device
)

# Visualize the training metrics for the full model
print("\n--- Training Plots ---\n")
helper_utils.plot_training_metrics(training_metrics)

<br>

After training the full model, you can analyze the results. But wait, something isn't right here. Your prototype model trained successfully, showing steady improvement. However, the performance on the full 15-class dataset seems to have hit a wall. What happened?

A close look at the plots reveals the problem. While the **Training Loss** consistently decreases, the **Validation Loss** drops for a while and then begins to rise and fluctuate. At the same time, the **Validation Accuracy** gets stuck, plateauing without making further significant progress. This is a classic case of **overfitting**.

Overfitting occurs when a model learns the training data *too well*, including its noise and specific quirks, instead of the general, underlying patterns that would help it perform on new, unseen data. The widening gap between your training and validation loss is a clear sign your model is memorizing the training set instead of learning to **generalize**.

You might wonder why this happened now and not with the 9-class prototype. The reason is the significant increase in **task complexity**. Distinguishing between 15 classes is much harder than 9, requiring the model to learn more subtle features. Faced with this harder challenge, your powerful CNN model found an easier path to lowering the training loss: it started to memorize the training data instead of learning to generalize.

This overfitting problem presents a realistic challenge, similar to what you'd encounter in a real-world project. In this module's graded assignment, you'll tackle this issue by making several updates to your entire pipeline to see if you can improve the model's ability to generalize.

In [ ]:
# ### Optional: Uncomment and run this cell to see the predictions made by the full model

# helper_utils.visualise_predictions(
#     model=trained_model, 
#     data_loader=val_loader, 
#     device=device, 
#     grid=(3, 5)
# )

## Conclusion

Congratulations on completing the lab! You have successfully navigated the entire machine learning pipeline, from data preparation to building, training, and analyzing your very own Convolutional Neural Network.

You've put theory into practice by building a CNN architecture capable of learning complex visual patterns. More importantly, you've experienced a realistic, iterative development workflow by first creating a successful prototype and then scaling up to a more complex model. This process led you to encounter and diagnose overfitting, a fundamental challenge that every machine learning practitioner must learn to solve.

The skills you've developed here have prepared you for the next step. You've identified the problem, and in the graded assignment, you'll get to solve it. Well done!